In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [15]:
df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv", index_col = 0)

## Data Cleaning and Processing
The [dataset](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset) shows the cleanliness and provides documentation on the column names and values, which will be used in this section

In [ ]:
# https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset

print(f"Number of Rows: {len(df)}")

feature_cols = [
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

# drop any null rows
df = df.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Number of Clean Rows: {len(df)}")

Number of Rows: 114000
Number of Clean Rows: 114000


### Column Values
As for processing though, certain audio features are scaled to between 0 and 1 so that it will not dominate distance calculations for nearest neighbors model
- `danceability`, `energy`, `speechiness`, `acousticness`, `instrumentalness`, `liveliness`, `valence` $\rightarrow$ float 0 - 11
- `key` $\rightarrow$ int 0 - 11
- `loudness` $\rightarrow$ float -49.53 - 4.53 dB
- `mode` $\rightarrow$ int 0 or 1
- `tempo` $\rightarrow$ float 0 - 243 bpm

### Tempo
According to [Krumhansl](https://cogsci.northwestern.edu/events/2016-2017-events/Krumhansl_2000.pdf), if the time between each beat is more than 1.5 seconds, then listeners have difficulty grouping sounds. As such, 40 BPM $60 / 1.5 = 40$ will be used as the bare minimum of what is considered "music". Unfortunately, this column is machine generated and Spotify's algorithm has trouble classifying BPM for all songs accurately. I will be dropping songs below 40 BPM

### Key
Treating Key as linear is not feasible because it wraps around. 0 (C) and 11 (B) are adjacent semitones (just 1 key apart on the piano), but are treated as maximally distant from one another. Although musical key definitely contributes to how music sounds, I'll be dropping it because the harmonics are less important than the other features

### Loudness
Loudness has a negative skew because decibels are a logarithmic scale. I will use percentile-based clipping for loudness

### Tempo Doubling/Halving
Some songs might have their BPMs halved while others doubled. For example, 120 BPM might be represented as 60, 120, or even 240 BPM. While I love music, I will not be listening to the 100k+ songs to clean the bpm data. I do think bpm is still important enough to keep in, despite the possible discrepancies.

### Instrumentalness, Acousticness, Speechiness, Liveliness, Energy
Instrumentalness, Acousticness, Speechiness, Liveliness are all positively skewed. Energy is negatively skewed. I will apply a transformation to each feature to each one

### Mode
Mode is binary (0 = minor, 1 = major), which can represent a large swing in distance for a nearest neighbors model. 

In [19]:
# Filter out songs with missing or invalid tempo
df = df[df["tempo"] > 40.0].reset_index(drop=True)

print(f"Remaining valid tracks: {len(df):,}")

Remaining valid tracks: 113,826
